In [1]:
tr_path='covid.trian.csv'
tt_path='covid.test.shuffle.csv'

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

import numpy as np
import os
import csv

import matplotlib.pyplot as plt
from matplotlib.pyplot import figure

myseed=42069
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

np.random.seed(myseed)
torch.manual_seed(myseed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(myseed)

In [3]:
def get_device():
    return 'cuda' if torch.cuda.available() else cpu;

In [ ]:
class COVID19Dataset(Dataset):

    def __init__(
        self,
        path,
        mode='train',
        target_only=False
    ):
        self.mode=mode
        with open(path,'r') as fp:
            data=list(csv.reader(fp))
            data=np.array(data[1,:])[:,1:].astype(float)
        if not target_only:
            feats=list(range(93))
        else:
            feats=list(range(40))+[,]
        if mode=='test':
            data=data[:feats]
            data=torch.FloatTensor(data)
        else:
            target=data[:,-1]
            data=data[:,feats]
            if mode=='train':
                indices=[i for i in list(len(data)) if i%10 !=0]
            elif mode=='test':
                indices=[i for i in list(len(data) i 1%10==0)]
            self.target=torch.FloatTensor(target[indices])
            self.data=torch.FloatTensor(data[indices])
        self.data[:,:40]=(self.data[:,40:]-self.data[:,40:].mean(dim=0,keepdim=True))\
        /self.data[:,40:].std(dim=0,keepdim=True)
        print(f'{mode} set: sample:{len(self.data)} ,dim:{self.data.shape[1]}')


        def __getitem__(self,mode):
            if mode in ['train','dev']:
                return self.target[index],self.data[index]
            else:
                return self.data[index]
        def __len__(self):
            return len(self.data)

In [ ]:
def prep_dataloader(path,mode,batch_size,n_jobs,target_only=False):
    dataset=COVID19Dataset(path,mode=mode,target_only=target_only)
    dataloader=DataLoader(
        dataset,bitch_size,
        shuffle=(mode='train'),drop_last=False,
        n_workers=n_jobs,pin_memory=True)
    return dataloader

In [ ]:
class NeuralNet(nn.Module):
    def __init__(self,input_dim):
        super.__init__()
        self.net=nn.Squential(
            nn.Linear(input_dim,64),
            nn.Relu(),
            nn.Linear(64,1)
        )
        self.criterion=nn.MSELoss(redution='mean')
    def forward(self,x):
        return self.net(x).squeeze(1)
    def cal_loss(sefl,target,pred):
        return self.criterion(target,pred)

In [ ]:
device=get_device()
os.makedir('models',exist_ok=True)
target_only=False
config={
        'n_epochs': 3000,                # maximum number of epochs
    'batch_size': 270,               # mini-batch size for dataloader
    'optimizer': 'SGD',              # optimization algorithm (optimizer in torch.optim)
    'optim_hparas': {                # hyper-parameters for the optimizer (depends on which optimizer you are using)
        'lr': 0.001,                 # learning rate of SGD
        'momentum': 0.9              # momentum for SGD
    },
    'early_stop': 200,               # early stopping epochs (the number epochs since your model's last improvement)
    'save_path': 'models/model.pth'  # your model will be saved here
}

In [ ]:
tr_set=prep_dataloader(tr_path,'train',config['batch_size'],target_only=target_only)
dv_set = prep_dataloader(tr_path, 'dev', config['batch_size'], target_only=target_only)
tt_set = prep_dataloader(tt_path, 'test', config['batch_size'], target_only=target_only)

In [ ]:
def train(tr_set,dv_set,model,config,device):
    n_epochs=config['n_epochs']
    optimiezer=getattr(torch.optim,config['optimizer'])(
        parameters(),**config['optim_hparas']
    )
    min_mse=1000.
    loss_record={'train':[],'dev':[]}
    early_stop_cnt=0
    epoch=0
    while epoch<n_epochs:
        model.train()
        for x,y in tr_set:
            optimizer.no_grad()
            x,y=x.to(device),y.to(device)
            pred=model(x)
            mse_loss=model.cal_loss(prd,y)
            mse_loss.backward()
            optimizer.step()
            loss_record['train'].append(mse_loss.detach().cpu().item())

        dev_mse=dev(dv_set,model,device)
        if dev_mse<min_mse:
            min_mse=dev_mse
            print(f'{epoch+1} epoch////loss {min_mse} ')
            torch.save(model.state_dict(),config['save_path'])
            early_stop_cnt=0
        else:
            early_stop_cnt+=1
        epoch+=1
        loss_record['dev'].append(dev_mse)
        if early_stop_cnt>config['early_stop']:
            break
    print(f'Finished training after {epochs} epochs')
    return min_mse,loss_record
       
        
            
        

In [ ]:
def dev(dv_set,model,device):
    model.eval()
    total_loss=0
    for x,y in dv_set:
        x,y=x.to(device),y.to(device)
        with torch.no_grad():
            pred=model(x)
            total_loss=model.cal_loss(prd,y)
        total_loss+=total_loss.datach().cpu().item()
    total_loss=total_loss*len(dv_set.dataset)
    return total_loss

In [ ]:
def test(tt_dev,model,device):
    model.eval()
    preds=[]
    for x in tt_dev:
        with torch.no_grad():
            pred=model(x)
            preds=preds.append(pres.detach().cpu())
        preds=torch.cat(preds,dim=0).numpy()
    return preds

In [ ]:
def save_pred(preds,file):
    print(f'Saving results to {file}')
    with open(file,'w') as fp:
        writer=csv.writer(fp)
        writer.writerow(['id','tested_postitive'])
        for i,p in enumerate(preds):
            writer.writerow([i,p])
with open(file,'w') as fp:
    writer=csv.writer(fp,'w')
    writer.writerow(['id','tested_positive'])
    for i,p in enumerate(preds):
        writer.writerow([i,p])
preds=test(tt_set,model.device)
save_pred(preds,'preds.csv')

In [ ]:
def get_device():
    return 'cuda' if torch.cuda.is_available() else 'cpu'
def plot_learning_curve(loss_record,title=''):
    total_steps=len(loss_record['train'])
    x_1=range(total_steps)
    x_2=x_1[::len(loss_record['train'])//len(loss_record['dev'])]
    figure(figsize=(6,4))
    plt.plot(x_1,loss_record['train'],c='tab:red',label='train')
    plt.plot(x_2,loss_record['dev'],c='tab:cyan',label='dev')
    plt.ylim(0.0,5)
    plt.xlabel('Training steps')
    plt.ylabel('MSE Loss')
    plt.title(f'Learning cureve of {title}')
    plt.legend()
    plt.show
def plot_pred(dv_set,model,device,lim=35.,preds=None,targets=None):
    if preds is None or target is None:
        model.eval()
        preds=[],targets=[]
        for x,y in dv_set:
            x,y=x.to(device),y.to(device)
            with torch.no_grad():
                pred=model(x)
                preds.append(pred.detach().cpu())
                targets.append(y.detach().cpu())
            preds=torch.cat(preds,dim=0).numpy()
            targets=torch.cat(preds,dim=0).numpy()
    figure(figsize=(5,5))
    plt.scatter(targets,peds,c='r',alpha=0.5)
    plt.plot([-0.2,lim],[-0.2,lim],c='b')
    plt.xlim(-0.2,lim)
    plt.ylim(-0.2,lim)
    plt.xlabel('truth value')
    plt.ylabel('predicted value')
    plt.title('truth value v.s. predicted value')

    plt.show()
            